- do any despeckling, do it before converting to dB
- for hackathon, run in the unstable sandbox, as the packages there are the up to date ones that will work with the SAR stuff

In [ ]:
%pip uninstall odc-stac -y
%pip install odc-stac -q

%pip uninstall pystac-client -y
%pip install pystac-client -q
%pip install -U pystac-client -q


In [ ]:

%pip uninstall dea-tools -y
%pip install dea-tools -q

%pip install scikit-image -q


In [ ]:
#!pip list


In [ ]:
import os
import pathlib
import numpy as np
import xarray as xr
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import odc.geo.xr
import datacube

from scipy.ndimage import uniform_filter
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from matplotlib.colors import ListedColormap
from odc.geo import BoundingBox
from odc.geo.geom import Geometry


from pystac_client import Client
from odc.stac import load, configure_s3_access
from dea_tools.dask import create_local_dask_cluster

# for loading landsat

from dea_tools.datahandling import load_ard
from dea_tools.plotting import rgb
from dea_tools.classification import sklearn_flatten, sklearn_unflatten
from dea_tools.validation import xr_random_sampling
from dea_tools.landcover import lc_colourmap, get_colour_scheme, make_colourbar


In [ ]:
dc = datacube.Datacube(env="dev", app="S1_Backscatter_Demo")


In [ ]:
# becaause I'm working in the devbox with a sandbox kernel I can't "find" the src folder so I've copied the speckle filter code here

# Adapted from https://stackoverflow.com/questions/39785970/speckle-lee-filter-in-python
def lee_filter(img, size):
    """
    Applies the Lee filter to reduce speckle noise in an image.

    Parameters:
    img (ndarray): Input image to be filtered.
    size (int): Size of the uniform filter window.

    Returns:
    ndarray: The filtered image.
    """
    img_mean = uniform_filter(img, size)
    img_sqr_mean = uniform_filter(img**2, size)
    img_variance = img_sqr_mean - img_mean**2

    overall_variance = np.var(img)

    img_weights = img_variance / (img_variance + overall_variance)
    img_output = img_mean + img_weights * (img - img_mean)
    return img_output


# Define a function to apply the Lee filter to a DataArray
def apply_lee_filter(data_array, size=7):
    """
    Applies the Lee filter to the provided DataArray.

    Parameters:
    data_array (xarray.DataArray): The data array to be filtered.
    size (int): Size of the uniform filter window. Default is 7.

    Returns:
    xarray.DataArray: The filtered data array.
    """
    data_array_filled = data_array.fillna(0)
    filtered_data = xr.apply_ufunc(
        lee_filter,
        data_array_filled,
        kwargs={"size": size},
        input_core_dims=[["y", "x"]],
        output_core_dims=[["y", "x"]],
        dask_gufunc_kwargs={"allow_rechunk": True},
        vectorize=True,
        dask="parallelized",
        output_dtypes=[data_array.dtype],
    )
    return filtered_data


In [ ]:
# environment set up

catalog = "https://explorer.dev.dea.ga.gov.au/stac"
stac_client = Client.open(catalog)
configure_s3_access(cloud_default = True, aws_unsigned = True)
client = create_local_dask_cluster(return_client=True)


In [ ]:
region_codes = ["x148y166"]

year = "2024"
start_date = f"{year}-01-01"
end_date = f"{year}-12-01"
time = (start_date, end_date)


In [ ]:
# open tiles and select

gdf = gpd.read_file(
    "~/gdata1/projects/fc-sub-annual/data/testing_minitile_suite.geojson"
)

gdf = gdf[gdf["region_code"].isin(region_codes)]
geom = Geometry(geom=gdf.iloc[0].geometry, crs=gdf.crs)

# for stac:
minx, miny, maxx, maxy = gdf.total_bounds
bbox = [minx, miny, maxx, maxy]
geom_bbox = BoundingBox(left=minx, bottom=miny, right=maxx, top=maxy, crs="EPSG:4326")


In [ ]:
product_to_load = "ga_s1_nrb_iw_vv_vh_0"
measurements_to_load = ["VV_gamma0", "VH_gamma0", "mask"]
output_crs = "EPSG:3577"
output_res = 20


In [ ]:
# using datacube to load multispec data so I can use load_ard for masking
ds_s2 = load_ard(
    dc = dc,
    products = ["ga_s2am_ard_3", "ga_s2bm_ard_3"],
    measurements = ["nbart_red", "nbart_green", "nbart_blue", "nbart_nir_1"],
    cloud_mask = "s2cloudless",
    mask_pixel_quality = True,
    mask_contiguity = True,
    skip_broken_datasets = True,
    time = (start_date, end_date),
    resolution = (-20, 20),
    geopolygon = geom_bbox.boundary(),
    output_crs = "EPSG:3577",
    dask_chunks = {},
    
)


In [ ]:
# make S2 monthly medians for use later on and for comparing to S1
ds_s2_month = ds_s2.groupby("time.month").median("time", keep_attrs = True)


In [ ]:
# load SAR with datacube
ds_s1 = dc.load(
    product=product_to_load,
    measurements=measurements_to_load,
    time = (start_date, end_date),
    resolution= (-20, 20),
    output_crs = "EPSG:3577",
    geopolygon = geom_bbox.boundary(),
    group_by = "solar_day",
    dask_chunks = {}
)


The below code loads the SAR data using 0dc.stac. This worked for me when I was using the unstable sandbox but threw an error (sent to Caitlin) when I tried to move to the stable sandbox.

In [ ]:
# items_s1 = stac_client.search(
#     collections = [product_to_load],
#     datetime = f"{start_date}/{end_date}",
#     bbox = geom_bbox,
# ).item_collection()

# print(f"Found {len(items_s1)} items")

# # view by orbit state

# ascending = [item for item in items_s1 if item.properties["sat:orbit_state"]=="ascending"]
# print(f"Number of items with ascending orbit state: {len(ascending)}")

# descending = [item for item in items_s1 if item.properties["sat:orbit_state"]=="descending"]
# print(f"Number of items with descending orbit state: {len(descending)}")

# #------- no mixed orbits so just group by solar day -------

# ds_s1 = load(
#     collections = [product_to_load],
#     bands = measurements_to_load,
#     items = items_s1,
#     intersects = geom_bbox.boundary(),
#     crs = output_crs,
#     resolution = output_res,
#     groupby = "solar_day",
#     chunks = {}
# )


In [ ]:
ds_s1 = ds_s1.compute()


based on some discussion in Teams, mean is better for SAR than median, and it should be done before converting to db. This may also mean I don't need to do a speckle filter.
Will apply the mask first, and then compare speckly filter and monthly means.

In [ ]:
ds_s1["VV_gamma0_masked"] = xr.where(ds_s1.mask==0, ds_s1["VV_gamma0"], np.nan)
ds_s1["VH_gamma0_masked"] = xr.where(ds_s1.mask==0, ds_s1["VH_gamma0"], np.nan)


In [ ]:
ds_s1["VV_gamma0_masked_filtered"] = apply_lee_filter(ds_s1["VV_gamma0_masked"], size=5)
ds_s1["VH_gamma0_masked_filtered"] = apply_lee_filter(ds_s1["VH_gamma0_masked"], size=5)


In [ ]:
# groupby month

ds_s1_month = ds_s1.groupby("time.month").mean("time", keep_attrs = True)


In [ ]:
# before and after speckle filtering

fig, axes = plt.subplots(1, 2, figsize=(10, 5), layout="constrained", subplot_kw={"projection": ccrs.epsg(3577)})

ds_s1_timestep = ds_s1_month.isel(month=0)

vmin = float(ds_s1_timestep["VV_gamma0_masked"].quantile(0.02))
vmax = float(ds_s1_timestep["VV_gamma0_masked"].quantile(0.98))

im0 = ds_s1_timestep["VV_gamma0_masked"].plot(
    ax=axes[0], cmap="Greys_r", robust=True, transform=ccrs.epsg(3577),
    add_colorbar=False, vmin=vmin, vmax=vmax
)
axes[0].set_title("Monthly mean, no speckle filtering")
gl0 = axes[0].gridlines(draw_labels=True)
gl0.xlines = False
gl0.ylines = False
gl0.right_labels = False
gl0.top_labels = False

im1 = ds_s1_timestep["VV_gamma0_masked_filtered"].plot(
    ax=axes[1], cmap="Greys_r", robust=True, transform=ccrs.epsg(3577),
    add_colorbar=False, vmin=vmin, vmax=vmax
)
axes[1].set_title("Monthly mean, with speckle filtering")
gl1 = axes[1].gridlines(draw_labels=True)
gl1.xlines = False
gl1.ylines = False
gl1.right_labels = False
gl1.top_labels = False


cbar = fig.colorbar(im1, ax=axes, orientation="horizontal", fraction=0.05, pad=0.07, aspect=30)
plt.show()


In [ ]:
# make ratio of VV to VH band
ds_s1_month["VH_over_VV_gamma0_masked"] = ds_s1_month["VH_gamma0_masked"] / ds_s1_month["VV_gamma0_masked"]
ds_s1_month["VH_over_VV_gamma0_masked_filtered"] = ds_s1_month["VH_gamma0_masked_filtered"] / ds_s1_month["VV_gamma0_masked_filtered"]


In [ ]:
# converting to dB, simplify data array name for future steps

ds_s1_month["VV"] = 10*np.log10(ds_s1_month["VV_gamma0_masked"])
ds_s1_month["VH"] = 10*np.log10(ds_s1_month["VH_gamma0_masked"])
ds_s1_month["VH_over_VV"] = 10*np.log10(ds_s1_month["VH_over_VV_gamma0_masked"])

ds_s1_month["VV_filtered"] = 10*np.log10(ds_s1_month["VV_gamma0_masked_filtered"])
ds_s1_month["VH_filtered"] = 10*np.log10(ds_s1_month["VH_gamma0_masked_filtered"])
ds_s1_month["VH_over_VV_filtered"] = 10*np.log10(ds_s1_month["VH_over_VV_gamma0_masked_filtered"])



In [ ]:
# scale data to the 98th percentiles
data_min = ds_s1_month.quantile(0.02)
data_max = ds_s1_month.quantile(0.98)
data_scaled = (ds_s1_month - data_min) / (data_max - data_min)


### Make monthly medians for VV, VH and landsat 8, do PCA on them and try throwing it into a random forest

In [ ]:
rgb(data_scaled, bands=["VV_filtered", "VH_filtered", "VH_over_VV_filtered"], col="month", col_wrap=3, size=4)


## prepare data for PCA and do PCA


In [ ]:

ds_s2_month.compute()


In [ ]:
pca_xr = data_scaled.drop_vars(["VV_gamma0",
                                "VH_gamma0",
                                "VV_gamma0_masked", 
                                "VH_gamma0_masked", 
                                "mask", 
                                "VV_gamma0_masked_filtered", 
                                "VH_gamma0_masked_filtered", 
                                "VH_over_VV_gamma0_masked", 
                                "VH_over_VV_gamma0_masked_filtered",
                                "VV_filtered",
                                "VH_filtered",
                                "VH_over_VV_filtered"])

pca_xr_filtered = data_scaled.drop_vars(["VV_gamma0", 
                                "VH_gamma0",
                                "VV_gamma0_masked", 
                                "VH_gamma0_masked", 
                                "mask", 
                                "VV_gamma0_masked_filtered", 
                                "VH_gamma0_masked_filtered", 
                                "VH_over_VV_gamma0_masked", 
                                "VH_over_VV_gamma0_masked_filtered",
                                "VV",
                                "VH",
                                "VH_over_VV"])



In [ ]:
if 'month' in pca_xr.dims:
    pca_xr = pca_xr.rename({'month': 'time'})
if 'month' in pca_xr_filtered.dims:
    pca_xr_filtered = pca_xr_filtered.rename({'month': 'time'})
if 'month' in ds_s2_month.dims:
    ds_s2_month = ds_s2_month.rename({'month': 'time'})


In [ ]:
x_s1 = sklearn_flatten(pca_xr_filtered)
pca_s1 = PCA(n_components=4)


In [ ]:
pca_s1.fit(x_s1)
print("Relative variance in principal components:", pca_s1.explained_variance_ratio_)


In [ ]:
predict_s1 = pca_s1.transform(x_s1)


In [ ]:
out_s1 = sklearn_unflatten(predict_s1, pca_xr_filtered)
out_s1 = out_s1.to_dataset(dim = out_s1.dims[0]).transpose("time", "y", "x")


In [ ]:
rgb(out_s1, bands=[0, 1, 2], col="time", col_wrap=3, size=4)


## Random sampling to build training dataset from land cover 2.0

In [ ]:
lc = dc.load(
    product = "ga_ls_landcover_class_cyear_3",
    measurements = ["level3"],
    resolution = (-30,30), # make this a multiple of 20
    output_crs = "EPSG:3577",
    group_by = "solar_day",
    time = (year),
    geopolygon = geom_bbox.boundary(),
    dask_chunks = {}
)


In [ ]:
lc = lc["level3"].squeeze()
lc = lc.where(lc != 255)  # Mask out no data values


In [ ]:
# Equal stratified random samples
train_points_equal_stratified_random = xr_random_sampling(
    lc, sampling="equal_stratified_random", n=1000
)


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(20, 6), sharey=True)

colour_scheme = get_colour_scheme("level3")
cmap, norm = lc_colourmap(colour_scheme)

# Trim no-data off the colormap
cmap2=ListedColormap(cmap.colors[:-1])

ds_s2_month[["nbart_red", "nbart_green", "nbart_blue"]].isel(time=0).to_array().plot.imshow(
    robust=True, ax=ax[0], add_labels=False
)
train_points_equal_stratified_random.plot(ax=ax[0], column="class", cmap=cmap2, legend=True, categorical=True)

ax[0].set_title("RGB Sentinel-2 Median, month 0")
ax[0].axes.get_xaxis().set_ticks([])
ax[0].axes.get_yaxis().set_ticks([])

pca_xr_filtered[["VV", "VH", "VH_over_VV"]].isel(time=0).to_array().plot.imshow(
    robust=True, ax=ax[1], add_labels=False
)
ax[1].set_title("RGB Sentinel-1 Median, month 0 (VV, VH, VH over VV)")
ax[1].axes.get_xaxis().set_ticks([])
ax[1].axes.get_yaxis().set_ticks([])


im = lc.plot(cmap=cmap, norm=norm, ax=ax[2], add_labels=False, add_colorbar=False)
make_colourbar(fig, ax[2], measurement="level3", labelsize=7, horizontal=False)

ax[2].set_title("DEA Land Cover")
ax[2].axes.get_xaxis().set_ticks([])
ax[2].axes.get_yaxis().set_ticks([]);


## Combine original bands, PCA and GLCM to use for random forrest

In [ ]:
combined_inputs = xr.merge([pca_xr_filtered, out_s1])


In [ ]:
lc_samples = train_points_equal_stratified_random.copy()
lc_samples['x'] = lc_samples.geometry.x
lc_samples['y'] = lc_samples.geometry.y


In [ ]:
def extract_features_at_points(xr_data, x_coords, y_coords):
    features = []
    for xi, yi in zip(x_coords, y_coords):
        vals = []
        for var in xr_data.data_vars:
            vals.extend(xr_data[var].sel(x=xi, y=yi, method='nearest').values.flatten())
        features.append(vals)
    return np.array(features)


In [ ]:
lc_samples = train_points_equal_stratified_random.copy()
lc_samples['x'] = lc_samples.geometry.x
lc_samples['y'] = lc_samples.geometry.y

# Flatten the combined_inputs grid for ML
X_grid_full = sklearn_flatten(combined_inputs)

# Get the x, y coordinates of the grid
x_coords = combined_inputs['x'].values
y_coords = combined_inputs['y'].values

# Build a mapping from (x, y) to flattened index
def get_flat_index(xi, yi, x_coords, y_coords):
    x_idx = np.abs(x_coords - xi).argmin()
    y_idx = np.abs(y_coords - yi).argmin()
    return y_idx * len(x_coords) + x_idx

# Get indices for training points
indices = [get_flat_index(xi, yi, x_coords, y_coords) for xi, yi in zip(lc_samples['x'], lc_samples['y'])]

# Extract features for training points from the flattened grid
X = X_grid_full[indices]
y = lc_samples['class'].values

# Train/validation split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Train random forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

val_score = rf.score(X_val, y_val)
print(f"Validation accuracy: {val_score:.3f}")

# Predict over all data arrays
y_pred_grid = rf.predict(X_grid_full)

# Unflatten to raster
raster_pred = sklearn_unflatten(y_pred_grid, combined_inputs)
raster_pred = xr.DataArray(raster_pred, dims=combined_inputs.dims, coords=combined_inputs.coords, name='rf_class')


# Plot
raster_pred.plot.imshow(x='x', y='y', col='time', cmap='tab10', col_wrap=3, size=4)
plt.show()


In [ ]:
# X_grid_full = sklearn_flatten(combined_inputs)

# x_coords = combined_inputs['x'].values
# y_coords = combined_inputs['y'].values

# # Build a mapping from (x, y) to flattened index
# def get_flat_index(xi, yi, x_coords, y_coords):
#     # Find nearest indices
#     x_idx = np.abs(x_coords - xi).argmin()
#     y_idx = np.abs(y_coords - yi).argmin()
#     # Calculate flat index (row-major order: y changes fastest)
#     return y_idx * len(x_coords) + x_idx

# # Get indices for training points
# indices = [get_flat_index(xi, yi, x_coords, y_coords) for xi, yi in zip(lc_samples['x'], lc_samples['y'])]

# # Extract features for training points from the flattened grid
# X = X_grid_full[indices]
# y = lc_samples['class'].values

# X = extract_features_at_points(combined_inputs, lc_samples['x'], lc_samples['y'])
# y = lc_samples['class'].values

# #Train/validation split
# X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# # Train random forest
# rf = RandomForestClassifier(n_estimators=100, random_state=42)
# rf.fit(X_train, y_train)

# val_score = rf.score(X_val, y_val)
# print(f"Validation accuracy: {val_score:.3f}")

# # Predict over all data arrays
# # Make sure to use the same feature extraction as for training

# y_pred_grid = rf.predict(X_grid_full)

# # Unflatten to raster
# raster_pred = sklearn_unflatten(y_pred_grid, combined_inputs)
# raster_pred = xr.DataArray(raster_pred, dims=combined_inputs.dims, coords=combined_inputs.coords, name='rf_class')

# # Plot
# raster_pred.plot.imshow(x='x', y='y', col='time', cmap='tab20')
# plt.title('Random Forest Classification Result')
# plt.show()
